# Cross-Dataset Benchmarking & Dynamic Weight Inspection
### Interactive Analysis Notebook for Adaptive Fusion Thesis

In [ ]:
import sys, os
sys.path.append("..")
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.detector import AdaptiveFusionDetector
from src.data.dataset import MultiBranchDeepfakeDataset
from src.evaluation.evaluator import CrossDatasetEvaluator
from src.evaluation.weight_analyzer import FusionWeightAnalyzer
from src.utils.visualizer import ForensicVisualizer

In [ ]:
# Load Model & Benchmark Datasets
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AdaptiveFusionDetector(feature_dim=128, fusion_type="domain_conditioned").to(device)

eval_datasets = {
    "FaceForensics++ (HQ)": MultiBranchDeepfakeDataset(num_synthetic_samples=20),
    "Celeb-DF v2": MultiBranchDeepfakeDataset(num_synthetic_samples=20),
    "DFDC": MultiBranchDeepfakeDataset(num_synthetic_samples=20),
    "Diffusion OOD": MultiBranchDeepfakeDataset(num_synthetic_samples=20),
    "LivePortrait Reenactment OOD": MultiBranchDeepfakeDataset(num_synthetic_samples=20),
}

evaluator = CrossDatasetEvaluator(model, device=device)
results_df = evaluator.run_benchmark_suite(eval_datasets, batch_size=8)
print(results_df.to_string(index=False))

In [ ]:
# Inspect Weight Shift under JPEG Compression
analyzer = FusionWeightAnalyzer(model, device=device)
shift_df = analyzer.analyze_compression_shift(eval_datasets["FaceForensics++ (HQ)"], quality_levels=[95, 75, 50, 25])
fig = analyzer.plot_compression_shift(shift_df)
plt.show()